# 02 — Fund NAV and flows

**The question:** *Did this fund's investors put money in or take it out over
the last two years — and did the quota keep up?*

Net assets move for two entirely different reasons: the portfolio made or lost
money, and investors subscribed or redeemed. Reading a rising NAV as performance
is one of the easiest mistakes to make with fund data, and CVM publishes both
halves separately so you do not have to guess.

Endpoints: `search_funds`, `fund_profile`, `fund_nav`, `panel`.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
COVERAGE = as_of("fund_nav", "funds_fi", "funds_fip")

Three things to take from that table before going further.

1. `funds_fi.as_of` is one month **ahead** of `complete_through`. CVM publishes
   with a one-to-two month lag, so the newest month that has landed is only
   fractionally filed. Default windows stop at `complete_through` on purpose.
2. `fund_nav.newest_period` is **2026-12-31** — a FIP year-end key that has not
   happened. Never read it as freshness.
3. The `fund_nav` note is the applicability rule: a null outside a family's
   column set is *not applicable*, not missing.

## Find a fund

`search_funds` is a name search over the registry. It is **clamped, not
refused**: 25 rows anonymous, 200 signed in. Asking for more silently gives you
the ceiling, so check how many you got rather than assuming.

In [ ]:
hits = silo.search_funds("BRADESCO", limit=200)   # 200 is the SIGNED-IN ceiling
print(f"asked for 200, received {len(hits)} — tier is {silo.tier!r}")
print()
pd.DataFrame(hits)[["cnpj", "entity_type", "fund_name", "latest_aum"]].head(8)

## Registry facts: `fund_profile`

In [ ]:
CNPJ = "32312124000107"   # BRADESCO FIF - CIC RENDA FIXA REFERENCIADA DI MAX

profile = silo.fund_profile(CNPJ)[0]
for k, v in profile.items():
    print(f"  {k:20s} {v}")

## The honest default window

This is the part of `fund_nav` worth understanding before you plot anything.

With **no explicit `end`**, the series stops at the family's latest **complete**
period. A month CVM has only fractionally published is withheld rather than
served as if it were the whole industry.

Pass an **explicit `end`** and you get the window verbatim, partial months
included. Both behaviours are correct; only one of them is safe to chart without
a caveat.

In [ ]:
honest = silo.fund_nav(CNPJ, start="2024-09-01")
verbatim = silo.fund_nav(CNPJ, start="2024-09-01", end="2026-12-31")

print(f"default window  : {len(honest):>3} rows, last period {honest[-1]['period']}")
print(f"explicit end    : {len(verbatim):>3} rows, last period {verbatim[-1]['period']}")
print()
extra = [r["period"] for r in verbatim][len(honest):]
if extra:
    print(f"the explicit window adds {extra} — partially filed, and served verbatim")
    print(f"complete_through for funds_fi is {COVERAGE.loc['funds_fi', 'complete_through']}")
else:
    print("no partial month is outstanding right now; the two windows agree")

## Not applicable is not missing

`fund_nav` returns the same eleven columns for every family. This fund is an
`fi`, so it files `nav`, `quota`, `quotaholders`, `inflows` and `redemptions`.
The other columns come back `null` — **by construction**, carrying no
information whatsoever. They are not gaps, not late data, and not zero.

In [ ]:
nav = pd.DataFrame(silo.fund_nav(CNPJ, start="2024-09-01"))
nav["period"] = pd.to_datetime(nav["period"])
nav = nav.set_index("period").sort_index()

family = profile["entity_type"]
files = silo.catalog()["applicability"]["fund_nav"]["columns_by_family"][family]

metric_cols = ["nav", "quota", "quotaholders", "delinquency",
               "monthly_yield", "inflows", "redemptions", "assets"]
audit = pd.DataFrame({
    "applicable": [c in files for c in metric_cols],
    "non_null_rows": [nav[c].notna().sum() for c in metric_cols],
}, index=metric_cols)
audit["reading"] = [
    "blank in the filing" if a else "NOT APPLICABLE — set by construction"
    for a in audit["applicable"]
]
print(f"family: {family}   files: {', '.join(files)}\n")
print(audit.to_string())

## Flows and performance, side by side

`panel` is the primitive. One call, four metrics, long rows that pivot to a
matrix. Net flow is a **local** subtraction — the API does not serve it, because
it does not serve anything it did not receive from a filing.

In [ ]:
p = silo.panel([CNPJ], ["nav", "inflows", "redemptions", "quota"],
               freq="month", start="2024-09-01", wide=True)
p.columns = p.columns.droplevel(0)          # one id: drop it from the columns

p["net_flow"] = p["inflows"] - p["redemptions"]
p["nav_change"] = p["nav"].diff()
p["quota_return"] = p["quota"].pct_change()

p[["nav", "net_flow", "nav_change", "quota_return"]].tail(12)

Read the last two columns together. `nav_change` is what the fund's net assets
did; `quota_return` is what an investor who stayed put earned. Where they
disagree, the difference is investors arriving or leaving — which is exactly
what `net_flow` measures.

In [ ]:
tot_in = p["inflows"].sum()
tot_out = p["redemptions"].sum()
window = f"{p.index.min():%Y-%m} .. {p.index.max():%Y-%m}"

print(f"window                 : {window}")
print(f"gross subscriptions    : R$ {tot_in:>18,.0f}")
print(f"gross redemptions      : R$ {tot_out:>18,.0f}")
print(f"net flow               : R$ {tot_in - tot_out:>18,.0f}")
print()
print(f"NAV at start           : R$ {p['nav'].iloc[0]:>18,.0f}")
print(f"NAV at end             : R$ {p['nav'].iloc[-1]:>18,.0f}")
print(f"NAV change             : R$ {p['nav'].iloc[-1] - p['nav'].iloc[0]:>18,.0f}")
print()
print(f"quota, cumulative      : {100 * (p['quota'].iloc[-1] / p['quota'].iloc[0] - 1):>6.2f}%")
print()
print("The gap between 'NAV change' and what the quota earned is investor money")
print("moving, not performance. Neither number alone tells you which.")
print()
print(f"CAVEAT: gross inflows and redemptions are as filed. They are NOT netted")
print(f"at source, and a month with both is a month with churn, not a wash.")

## The grain has four parts, not three

`panel` rows are keyed on `(id, asset_class, date, metric)`. **385 CNPJs file
under two fund families in the same month** (`fi` and `fidc`), and the panel
returns one row per family for them.

Pivot on `(id, date, metric)` and you either raise on the duplicate or — worse —
silently average two different vehicles into one that does not exist. The SDK
refuses rather than average; `wide=False` keeps the four-part key visible.

In [ ]:
long_rows = silo.panel([CNPJ], ["nav"], freq="month",
                       start="2026-06-01", wide=False)
pd.DataFrame(long_rows)

`asset_class` is on every row. Pass `entity_type=` to keep one family, or keep
`asset_class` in your own pivot key. The SDK's `wide=True` checks for you:

```python
silo.panel(ids, metrics, wide=True)
# ValueError: panel is not unique on (id, date, metric): [...] file under
#             more than one asset_class in the window.
```

## Paging `fund_nav` needs a family

`fund_nav` pages with a `p_after` cursor like `quote_history` — but its cursor
is a **bare period**, and a period is unique only *within* one family. Paging a
two-family result on a bare date would skip or repeat a row at a page edge, so
the server refuses rather than answer wrongly.

The SDK checks before spending the round trip:

In [ ]:
try:
    next(silo.iter_fund_nav(CNPJ, entity_type=""))
except ValueError as exc:
    print("ValueError (local):", exc)

print()
rows = silo.fund_nav_all(CNPJ, entity_type="fi", start="2019-01-01")
print(f"paged whole history: {len(rows):,} months, "
      f"{rows[0]['period']} .. {rows[-1]['period']}")

Without `p_after` you get every family at once, each row labelled with its
`entity_type` — that mode needs no family and is what `fund_nav()` does.

## Where this goes next

* Notebook `03` opens the same fund up: what it actually holds.
* Notebook `04` does this for a FIDC, where one metric has a regime break in the
  middle of the window.
* To screen a **whole family** in one paged call, sign in and use `panel`
  universe mode: `panel_all(None, ["nav"], entity_type="fi", min_nav=...)`.
  Anonymous callers enumerate from the `funds` view instead.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.